# T2-D — Per-Event Candidate + Order-Constrained Ceiling Diagnostic

Known-source-video diagnostic only. D1 measures event-only candidate rank, D2 measures the all-event reference-window oracle path, and D3 measures the cost of forcing one event into its reference window. This is not a retrieval algorithm, T3, or end-to-end competition recall.

## INPUT CẦN GẮN TRÊN KAGGLE

1. Stage 1 index: `/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle`
2. Stage 1B encoder verification: `/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports`
3. Stage 1E language-path freeze: `/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze`
4. Offline OpenAI CLIP: `/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32`
5. Offline OPUS vi-en translator: `/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en`
6. Frozen RT2 benchmark: `/kaggle/input/datasets/irthn1311/triage-eg-rt2-ai-benchmark-bundle`

Nested roots are discovered by unique markers. The experiment performs no model download.

## INPUT KHÔNG CẦN

Raw AIC dataset/videos, Stage 0, T2 output bundle, M1, MB1/MB1-E1, OCR, ASR, Objects, VLM, Event Graph, Agent, images, or human-review assets.

## OUTPUT ZIP

`/kaggle/working/triage_eg_t2d_ceiling_bundle.zip`


In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path("/kaggle/working/AIC2026_TeamPTK_SGU")
if not (REPO_DIR / ".git").is_dir():
    clone_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
        env=clone_env,
    )
sys.path.insert(0, str(REPO_DIR / "src"))
COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
print("resolved commit:", COMMIT)

In [ ]:
STAGE1_INPUT = Path(
    os.environ.get(
        "AIC_STAGE1_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle"
    )
)
STAGE1B_INPUT = Path(
    os.environ.get(
        "AIC_STAGE1B_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports",
    )
)
STAGE1E_INPUT = Path(
    os.environ.get(
        "AIC_STAGE1E_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze",
    )
)
CLIP_INPUT = Path(
    os.environ.get("AIC_CLIP_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32")
)
OPUS_INPUT = Path(
    os.environ.get("AIC_OPUS_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en")
)
BENCHMARK_INPUT = Path(
    os.environ.get(
        "AIC_RT2_BENCHMARK_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-rt2-ai-benchmark-bundle",
    )
)
OUTPUT_ROOT = Path("/kaggle/working/triage_eg_t2d_ceiling")
ZIP_PATH = Path("/kaggle/working/triage_eg_t2d_ceiling_bundle.zip")
print(
    {
        "stage1": str(STAGE1_INPUT),
        "stage1b": str(STAGE1B_INPUT),
        "stage1e": str(STAGE1E_INPUT),
        "clip": str(CLIP_INPUT),
        "opus": str(OPUS_INPUT),
        "benchmark": str(BENCHMARK_INPUT),
        "output": str(OUTPUT_ROOT),
        "zip": str(ZIP_PATH),
    }
)

In [ ]:
SEARCH_ROOT = Path("/kaggle/input")


def find_marker_roots(root: Path, marker: str, max_depth: int = 6):
    matches, frontier = [], [(Path(root), 0)]
    while frontier:
        current, depth = frontier.pop(0)
        if (current / marker).is_file():
            matches.append(current.resolve())
            continue
        if depth < max_depth and current.is_dir():
            frontier.extend(
                (child, depth + 1) for child in sorted(current.iterdir()) if child.is_dir()
            )
    return sorted(set(matches))


def resolve_root(requested: Path, marker: str) -> Path:
    matches = find_marker_roots(requested, marker)
    if not matches:
        matches = find_marker_roots(SEARCH_ROOT, marker)
    if len(matches) != 1:
        raise RuntimeError(f"Expected one root containing {marker}; found {matches}")
    return matches[0]


def resolve_input_file(requested: Path, filename: str) -> Path:
    if requested.is_file() and requested.name == filename:
        return requested.resolve()
    roots = find_marker_roots(requested, filename)
    if not roots:
        roots = find_marker_roots(SEARCH_ROOT, filename)
    paths = sorted({(root / filename).resolve() for root in roots})
    if len(paths) != 1:
        raise RuntimeError(f"Expected one {filename}; found {paths}")
    return paths[0]


STAGE1_ROOT = resolve_root(STAGE1_INPUT, "stage1_summary.json")
STAGE1B_ROOT = resolve_root(STAGE1B_INPUT, "stage1b_summary.json")
STAGE1E_ROOT = resolve_root(STAGE1E_INPUT, "language_path_contract.json")
CLIP_ROOT = resolve_root(CLIP_INPUT, "checkpoint/ViT-B-32.pt")
OPUS_ROOT = resolve_root(OPUS_INPUT, "model/config.json")
BENCHMARK_PATH = resolve_input_file(BENCHMARK_INPUT, "rt2_ai_benchmark.jsonl")
print(
    json.dumps(
        {
            "stage1_root": str(STAGE1_ROOT),
            "stage1b_root": str(STAGE1B_ROOT),
            "stage1e_root": str(STAGE1E_ROOT),
            "clip_root": str(CLIP_ROOT),
            "opus_root": str(OPUS_ROOT),
            "benchmark_path": str(BENCHMARK_PATH),
        },
        indent=2,
    )
)

In [ ]:
from triage_eg.experiments.t2d_ceiling import T2DRunnerConfig, preflight_t2d
from triage_eg.retrieval.stage2 import config_from_yaml

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
STAGE2 = config_from_yaml(
    REPO_DIR / "configs/retrieval/stage2_operational_runtime.yaml",
    stage1_root=STAGE1_ROOT,
    stage1b_root=STAGE1B_ROOT,
    stage1e_root=STAGE1E_ROOT,
    clip_asset_root=CLIP_ROOT,
    translator_asset_root=OPUS_ROOT,
    output_root=OUTPUT_ROOT / "_stage2_control",
    stage1d_config=REPO_DIR / "configs/retrieval/stage1d_translation_ablation.yaml",
    build_git_commit=COMMIT,
)
CONFIG = T2DRunnerConfig(STAGE2, BENCHMARK_PATH, OUTPUT_ROOT)
PREFLIGHT = preflight_t2d(CONFIG)
print(json.dumps(PREFLIGHT, indent=2))

In [ ]:
from triage_eg.experiments.reference_rt2 import load_rt2_benchmark
from triage_eg.experiments.t2d_ceiling import run_t2d

QUERIES = load_rt2_benchmark(BENCHMARK_PATH)
RESULT = run_t2d(CONFIG, QUERIES)
print(json.dumps(RESULT["summary"], indent=2))

In [ ]:
import pandas as pd

failures = pd.read_json(OUTPUT_ROOT / "k5_failure_analysis.jsonl", lines=True)
display(
    failures[
        [
            "query_id",
            "event_id",
            "best_neighborhood_rank",
            "rank_percentile",
            "forced_event_relative_score_gap",
            "oracle_path_feasible",
            "oracle_relative_score_gap",
            "t2_k5_unique_anchor_count",
            "t2_k5_minimum_seconds_error",
        ]
    ]
)
print(json.dumps(RESULT["metrics"]["T2_REPRODUCTION"], indent=2))
print(json.dumps(RESULT["metrics"]["K5_PATH_DIVERSITY"], indent=2))

In [ ]:
from triage_eg.experiments.t2d_ceiling import create_t2d_bundle

bundle = create_t2d_bundle(OUTPUT_ROOT, ZIP_PATH)
print("T2D_IMPLEMENTATION_STATUS = COMPLETE")
print("T2D_REAL_STATUS = COMPLETE")
print("T2D_DIAGNOSTIC_STATUS = COMPLETE")
print("ROOT_CAUSE_DECISION = NOT_EVALUATED")
print("DOWNLOAD ZIP:", bundle, "size_bytes=", bundle.stat().st_size)